# Custom Tools Agent

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSV-AI/agent-playground/blob/main/notebooks/custom_tools.ipynb)

## Overview

This notebook demonstrates how to create a custom tools agent using PydanticAI with OpenRouter as the LLM provider. The agent can:
- Get the current time
- Retrieve user information
- Access company logos (as images)
- Process PDF documents

## Setup

Before running this notebook, make sure you have the required dependencies installed and your OpenRouter API key configured.

In [ ]:
# Install required dependencies
# Uncomment the line below if running in Google Colab or if dependencies are not installed
!pip install pydantic-ai pydantic python-dotenv

In [1]:
import os
from pydantic_ai import Agent, DocumentUrl, ImageUrl
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
from datetime import datetime
from pydantic import BaseModel

## Define Data Models

Define a Pydantic model for user data.

In [2]:
class User(BaseModel):
    name: str
    age: int

## Configure the LLM

Configure PydanticAI to use OpenRouter with OpenAI-compatible model.

In [3]:
# --- Configuration ---
# Since OpenRouter has a unified API, we can use the OpenAIModel with a custom provider.
OPENROUTER_MODEL = "openai/gpt-4o-mini"  # Using a placeholder compatible Gemma model

# 1. Configure the LLM for OpenRouter
# We use OpenAIModel because OpenRouter is OpenAI-compatible
# and we pass the OpenRouterProvider to configure the endpoint.
openrouter_model = OpenAIModel(
    OPENROUTER_MODEL,
    provider=OpenRouterProvider(
        api_key=os.environ.get("OPENROUTER_API_KEY") 
    ),
)

UserError: Set the `OPENROUTER_API_KEY` environment variable or pass it via `OpenRouterProvider(api_key=...)`to use the OpenRouter provider.

## Create the Agent

Initialize the agent with the configured LLM model.

In [ ]:
# 2. Create the Agent with the WebSearchTool
agent = Agent(
    model=openrouter_model,
)

## Define Custom Tools

Define custom tools that the agent can use to answer user queries.

In [ ]:
@agent.tool_plain
def get_current_time() -> datetime:
    return datetime.now()

@agent.tool_plain
def get_user() -> User:
    return User(name='John', age=30)

@agent.tool_plain
def get_company_logo() -> ImageUrl:
    return ImageUrl(url='https://iili.io/3Hs4FMg.png')

@agent.tool_plain
def get_document() -> DocumentUrl:
    return DocumentUrl(url='https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf')

## Run the Agent

Test the agent with various queries. It will use the custom tools to provide answers.

In [ ]:
# Execute a query to get the current time
result = agent.run_sync('What time is it?')
print(result.output)
#> The current time is 10:45 PM on April 17, 2025.

In [ ]:
# Execute a query to get the user name
result = agent.run_sync('What is the user name?')
print(result.output)
#> The user's name is John.

In [ ]:
# Execute a query to identify the company name from the logo
result = agent.run_sync('What is the company name in the logo?')
print(result.output)
#> The company name in the logo is "Pydantic."

In [ ]:
# Execute a query to get the main content of the document
result = agent.run_sync('What is the main content of the document?')
print(result.output)
#> The document contains just the text "Dummy PDF file."